In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

# Module 3: Point Operations

This is where the algebraic structure from Module 1 meets the curves from Module 2.
We need to define an operation that combines two curve points and produces a third
curve point — a group operation.

## 3.1 Why not just add the coordinates?

The first instinct: if $P = (x_1, y_1)$ and $Q = (x_2, y_2)$, why not define
$P + Q = (x_1 + x_2,\; y_1 + y_2)$?

Because the result almost certainly **isn't on the curve**. Both $P$ and $Q$
satisfy $y^2 = x^3 + 7$, but $(y_1 + y_2)^2 \neq (x_1 + x_2)^3 + 7$ in
general. You've left the set. Closure is broken. No group. No signature scheme.

So the question is: **what operation on two curve points is guaranteed to produce
another curve point?**

The answer comes from the geometry of the curve itself. A line through two points
on a cubic curve (degree 3) always hits the curve at exactly one more point — this
is a consequence of Bezout's theorem. "Draw a line, find the third intersection"
is the one natural operation that **stays on the curve by construction**.

The reflection step (negating the $y$-coordinate of the third intersection) is
what makes the operation satisfy associativity and commutativity — turning it
into a proper Abelian group.

So "point addition" isn't adding numbers. It's a geometric construction that
happens to obey the same abstract rules as regular addition.

## 3.2 Geometric Intuition

```
Point Addition P + Q:          Point Doubling 2P:

     ·  Q                           ·
    / \                            /|\
   /   \                          / | \
  P     \  ← secant line        P  | tangent line
   \     \                        \ |
    \     T                        \T
     \   /                          |
      \ /                           |
       R = P + Q  (reflect T)       R = 2P  (reflect T)
```

1. Draw a line through P and Q (or the tangent at P for doubling)
2. The line hits the curve at a third point T
3. Reflect T across the x-axis to get R = P + Q

## 3.2 The Formulas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a_curve, b_curve = 1, 1

def curve_y(x):
    rhs = x**3 + a_curve * x + b_curve
    mask = rhs >= 0
    y = np.full_like(x, np.nan)
    y[mask] = np.sqrt(rhs[mask])
    return y

def find_third_point(px, py, qx, qy):
    """Line through P and Q intersects y^2 = x^3 + ax + b at a third point."""
    if px == qx:
        return None, None
    lam = (qy - py) / (qx - px)
    nu = py - lam * px
    # x^3 - lam^2 * x^2 + ... = 0, roots sum to lam^2
    rx = lam**2 - px - qx
    ry = lam * rx + nu
    return rx, ry

# --- Two examples: Addition and Doubling ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

x_vals = np.linspace(-1.5, 4, 2000)
y_pos = curve_y(x_vals)

for ax in (ax1, ax2):
    ax.plot(x_vals, y_pos, 'b-', linewidth=2)
    ax.plot(x_vals, -y_pos, 'b-', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.3)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.grid(True, alpha=0.2)

# --- Left panel: P + Q ---
Px, Py = 0.0, 1.0
Qx, Qy = 1.0, np.sqrt(1 + 1 + 1)
Tx, Ty = find_third_point(Px, Py, Qx, Qy)
Rx, Ry = Tx, -Ty

lam = (Qy - Py) / (Qx - Px)
nu = Py - lam * Px
x_line = np.linspace(-1.5, Tx + 0.5, 200)
y_line = lam * x_line + nu

ax1.plot(x_line, y_line, 'g--', linewidth=1.5, alpha=0.7, label='secant line')
ax1.plot([Tx, Rx], [Ty, Ry], 'r:', linewidth=1.5, alpha=0.7, label='reflect')
ax1.plot(Px, Py, 'ko', markersize=10, zorder=10)
ax1.plot(Qx, Qy, 'ko', markersize=10, zorder=10)
ax1.plot(Tx, Ty, 's', color='gray', markersize=9, zorder=10)
ax1.plot(Rx, Ry, 'r*', markersize=15, zorder=10)
ax1.annotate('P', (Px, Py), textcoords="offset points", xytext=(-15, 8), fontsize=13, fontweight='bold')
ax1.annotate('Q', (Qx, Qy), textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold')
ax1.annotate('T', (Tx, Ty), textcoords="offset points", xytext=(8, -15), fontsize=13, color='gray')
ax1.annotate('R = P + Q', (Rx, Ry), textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold', color='red')
ax1.set_title('Point Addition: $P + Q$', fontsize=14)
ax1.set_xlim(-1.5, 4)
ax1.set_ylim(-5, 5)
ax1.legend(fontsize=10)

# --- Right panel: 2P (doubling via tangent) ---
Dx = 1.0
Dy = np.sqrt(Dx**3 + a_curve * Dx + b_curve)
# tangent slope: dy/dx from implicit differentiation: (3x^2 + a) / (2y)
lam_d = (3 * Dx**2 + a_curve) / (2 * Dy)
nu_d = Dy - lam_d * Dx
Tx2 = lam_d**2 - 2 * Dx
Ty2 = lam_d * Tx2 + nu_d
Rx2, Ry2 = Tx2, -Ty2

x_line2 = np.linspace(-1.5, Tx2 + 0.5, 200)
y_line2 = lam_d * x_line2 + nu_d

ax2.plot(x_line2, y_line2, 'g--', linewidth=1.5, alpha=0.7, label='tangent line')
ax2.plot([Tx2, Rx2], [Ty2, Ry2], 'r:', linewidth=1.5, alpha=0.7, label='reflect')
ax2.plot(Dx, Dy, 'ko', markersize=10, zorder=10)
ax2.plot(Tx2, Ty2, 's', color='gray', markersize=9, zorder=10)
ax2.plot(Rx2, Ry2, 'r*', markersize=15, zorder=10)
ax2.annotate('P', (Dx, Dy), textcoords="offset points", xytext=(-15, 8), fontsize=13, fontweight='bold')
ax2.annotate('T', (Tx2, Ty2), textcoords="offset points", xytext=(8, -15), fontsize=13, color='gray')
ax2.annotate('R = 2P', (Rx2, Ry2), textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold', color='red')
ax2.set_title('Point Doubling: $2P$', fontsize=14)
ax2.set_xlim(-1.5, 4)
ax2.set_ylim(-5, 5)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Left:  Line through P and Q hits curve at T. Reflect T → R = P + Q.")
print("Right: Tangent at P hits curve at T. Reflect T → R = 2P.")

In [ ]:
from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

In [ ]:
def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

## 3.3 Scalar Multiplication (Double-and-Add)

**Where we are:** We can add two points and double a point. That means we
can compute $G + G = 2G$, then $2G + G = 3G$, then $4G$, $5G$, and so on.
Repeating point addition $d$ times gives us $d \times G$ — and that's
exactly the one-way function from the intro: **private key $d$ in, public
key $P$ out.**

$$P = d \times G$$

This is the moment the one-way function happens. Forward (computing $P$
from $d$) is fast. Reverse (finding $d$ from $P$) is the discrete log
problem — infeasible on secp256k1.

But $d$ is a 256-bit number — up to $\approx 10^{77}$. Adding $G$ to
itself that many times would take longer than the age of the universe.
We need a shortcut.

**Double-and-add** uses the binary representation of $d$ to do it in
$O(\log d)$ operations — at most 256 doublings and 256 additions instead
of $10^{77}$ additions.

```
Example: 13 × P  (13 = 1101 in binary)

Step  Binary  Action             Result
─────────────────────────────────────────
  0   1       result += addend    P
      ─       addend = 2×addend   2P
  1   0       (skip add)          P
      ─       addend = 2×addend   4P
  2   1       result += addend    P + 4P = 5P
      ─       addend = 2×addend   8P
  3   1       result += addend    5P + 8P = 13P
```

In [ ]:
def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

## 3.4 Compressed Public Keys

**Where we are:** We can now compute $P = d \times G$ — a public key from a
private key. The public key $P$ is a point $(x, y)$ on secp256k1. Both $x$
and $y$ are 256-bit numbers, so storing both takes 64 bytes (plus a prefix
byte = 65 bytes). Every Bitcoin transaction includes the public key, so this
adds up.

**The trick:** Remember the vertical symmetry from Module 2? For any $x$,
the curve equation $y^2 = x^3 + 7$ has at most **two** solutions: $y$ and
$p - y$ — one even, one odd. So if you know $x$ and which of the two $y$
values it is (even or odd), you can reconstruct the full point. That cuts
the public key nearly in half:

```
Uncompressed: 65 bytes  [04 || x (32 bytes) || y (32 bytes)]
Compressed:   33 bytes  [02/03 || x (32 bytes)]
                         02 = even y,  03 = odd y
```

This is why every Bitcoin address you see uses compressed keys — same
security, half the space on the blockchain.

In [ ]:
def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")